# Phase 3: Rhetorical Roles: MLA decoder

This is the **Rhetorical Roles** part of the full `mla` notebook (`notebooks/CS781_mla_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

## Rhetorical Role Prediction

In [13]:
import os, json, math, time, random, pickle, collections
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from huggingface_hub import hf_hub_download
for n in ["tokenizer", "device", "HF_TOKEN", "HF_DATASET_REPO"]: assert n in globals(), f"'{n}' missing, run the setup cell"

P3_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_3"; os.makedirs(P3_DIR + "/submission", exist_ok=True)
WINDOW, MAX_SENT_TOK = 1024, 128
USE_CLASS_WEIGHTS, CLASS_W_CLIP = True, 10.0
PAD_ID = tokenizer.token_to_id("[PAD]"); NL_IDS = tokenizer.encode("\n").ids
def amp(): return torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)

# ---------------- data ----------------
def _load(f): return json.load(open(hf_hub_download(HF_DATASET_REPO, f"Phase_3_RR/{f}", repo_type="dataset", token=HF_TOKEN), encoding="utf-8"))
L2I = _load("Train/label_vocab.json"); LABELS = sorted(L2I, key=L2I.get); NC = len(LABELS)

def norm_docs(raw, domain):
    items = list(raw.items()) if isinstance(raw, dict) else raw
    docs = []
    for it in items:
        did, v = (it if isinstance(it, tuple) else (it["id"], it))
        text = v if isinstance(v, list) else v["text"]
        labels = v.get("labels") if isinstance(v, dict) else None
        d = dict(id=did, domain=domain, text=text, labels=labels)
        d["sent_ids"] = [tokenizer.encode(s.strip()).ids[:MAX_SENT_TOK] + NL_IDS for s in text]
        d["y"] = [L2I[l] for l in labels] if labels is not None else None
        if labels is not None: assert len(labels) == len(text)
        docs.append(d)
    return docs

train_docs = norm_docs(_load("Train/CL_train.json"), "CL") + norm_docs(_load("Train/IT_train.json"), "IT")
val_docs   = norm_docs(_load("Train/CL_dev.json"), "CL")   + norm_docs(_load("Train/IT_dev.json"), "IT")
test_docs  = norm_docs(_load("Test/CL_test.json"), "CL")   + norm_docs(_load("Test/IT_test.json"), "IT")
assert len({d["id"] for d in test_docs}) == len(test_docs), "test doc ids are not unique across domains"
for nm, ds in [("train", train_docs), ("val", val_docs), ("test", test_docs)]:
    print(f"{nm:5s} docs={len(ds)} sents={sum(len(d['text']) for d in ds)} "
          f"CL={sum(d['domain']=='CL' for d in ds)} IT={sum(d['domain']=='IT' for d in ds)}")

cnt = np.bincount([y for d in train_docs for y in d["y"]], minlength=NC).astype(float)
CLASS_W = torch.tensor(np.clip(np.sqrt(cnt.sum() / (NC * np.maximum(cnt, 1))), 0, CLASS_W_CLIP), dtype=torch.float32, device=device)
print("class weights:", {LABELS[i]: round(float(CLASS_W[i]), 2) for i in range(NC)})

# ---------------- context modes / windows ----------------
CTX = {"sentence": dict(max_sents=1, stride_frac=0.5),        # each sentence alone
       "local8":   dict(max_sents=8, stride_frac=0.5),        # up to 8 consecutive sentences
       "document": dict(max_sents=10**9, stride_frac=0.75)}   # as many sentences as fit in 1024 tokens
BW = {"sentence": 64, "local8": 32, "document": 8}            # windows per batch

def make_windows(sent_ids, max_sents, stride_frac, max_tokens=WINDOW):
    n, lens, wins, a = len(sent_ids), [len(x) for x in sent_ids], [], 0
    while True:
        b, tot = a, 0
        while b < n and tot + lens[b] <= max_tokens and (b - a) < max_sents: tot += lens[b]; b += 1
        wins.append((a, b))
        if b >= n: break
        c, acc, target = a, 0, max(1, int(stride_frac * tot))
        while c < b and acc < target: acc += lens[c]; c += 1
        a = max(c, a + 1)
    return wins

def all_windows(docs, mode): return [(di, a, b) for di, d in enumerate(docs) for a, b in make_windows(d["sent_ids"], **CTX[mode])]

def collate(wins, docs):
    seqs, spans, ys = [], [], []
    for di, a, b in wins:
        d = docs[di]; ids, sp = [], []
        for k in range(a, b): s = len(ids); ids += d["sent_ids"][k]; sp.append((s, len(ids)))
        seqs.append(ids); spans.append(sp)
        if d["y"] is not None: ys += d["y"][a:b]
    L = max(len(s) for s in seqs)
    x = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    for j, s in enumerate(seqs): x[j, :len(s)] = torch.tensor(s)
    return x, spans, (torch.tensor(ys) if ys else None)

# ---------------- model: backbone + sentence pooling + head ----------------
def pool(H, spans):
    jw, st, en = [], [], []
    for j, sp in enumerate(spans):
        for s, e in sp: jw.append(j); st.append(s); en.append(e)
    jw, st, en = [torch.tensor(t, device=H.device) for t in (jw, st, en)]
    last = H[jw, en - 1]                                                   # delimiter token: has seen the whole sentence
    cs = torch.cat([torch.zeros_like(H[:, :1]).float(), H.float().cumsum(1)], dim=1)
    mean = ((cs[jw, en] - cs[jw, st]) / (en - st).unsqueeze(1)).to(H.dtype)  # mean over the sentence's tokens
    return torch.cat([last, mean], -1)

class RRModel(nn.Module):
    """Decoder backbone (causal) -> per-sentence features [last token ; mean] -> linear head."""
    def __init__(self, lm, d_model, n_labels=NC, dropout=0.1):
        super().__init__()
        self.lm, self.drop, self.head = lm, nn.Dropout(dropout), nn.Linear(2 * d_model, n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
    def hidden(self, ids):
        lm = self.lm; x = lm.token_embedding(ids)
        for layer in lm.layers: x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
        return lm.norm(x)
    def forward(self, ids, spans): return self.head(self.drop(pool(self.hidden(ids), spans)))

# ---------------- metrics (own implementation) ----------------
def rr_scores(gold, pred):
    g, p = np.array(gold), np.array(pred); labs = sorted(set(g.tolist()) | set(p.tolist())); per = {}
    for l in labs:
        tp = int(((g == l) & (p == l)).sum()); fp = int(((g != l) & (p == l)).sum()); fn = int(((g == l) & (p != l)).sum())
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[LABELS[l]] = dict(p=pr, r=rc, f1=2 * pr * rc / (pr + rc) if pr + rc else 0.0, support=int((g == l).sum()))
    m = lambda k: float(np.mean([v[k] for v in per.values()]))
    return dict(macro_f1=m("f1"), macro_p=m("p"), macro_r=m("r"), acc=float((g == p).mean()), per=per)

# ---------------- prediction ----------------
@torch.no_grad()
def predict_probs(model, docs, mode):
    model.eval(); bw = BW[mode]; out = {}
    for d in docs:
        n = len(d["sent_ids"]); P = torch.zeros(n, NC, device=device); C = torch.zeros(n, device=device)
        wins = [(0, a, b) for a, b in make_windows(d["sent_ids"], **CTX[mode])]
        for i in range(0, len(wins), bw):
            chunk = wins[i:i + bw]; x, spans, _ = collate(chunk, [d])
            with amp(): lg = model(x.to(device), spans)
            p = F.softmax(lg.float(), -1); k = 0
            for _, a, b in chunk: P[a:b] += p[k:k + b - a]; C[a:b] += 1; k += b - a
        out[d["id"]] = (P / C.unsqueeze(1)).cpu().numpy()
    return out

def evaluate(model, docs, mode):
    P = predict_probs(model, docs, mode)
    G = {dm: ([], []) for dm in ("CL", "IT", "ALL")}
    for d in docs:
        pr = P[d["id"]].argmax(1).tolist()
        for key in (d["domain"], "ALL"): G[key][0].extend(d["y"]); G[key][1].extend(pr)
    res = rr_scores(*G["ALL"]); res["by_domain"] = {dm: rr_scores(*G[dm]) for dm in ("CL", "IT") if G[dm][0]}
    return res, P

def print_rr(res):
    print(f"ALL  macro-F1 {res['macro_f1']:.4f} | P {res['macro_p']:.4f} | R {res['macro_r']:.4f} | acc {res['acc']:.4f}")
    for dm, r in res["by_domain"].items(): print(f"  {dm}  macro-F1 {r['macro_f1']:.4f} | P {r['macro_p']:.4f} | R {r['macro_r']:.4f} | acc {r['acc']:.4f}")

def write_test_predictions(model, name, mode):
    P = predict_probs(model, test_docs, mode)
    allp = {d["id"]: [LABELS[i] for i in P[d["id"]].argmax(1)] for d in test_docs}
    for d in test_docs: assert len(allp[d["id"]]) == len(d["text"])
    for suffix, sel in [("all", allp), ("CL", {d["id"]: allp[d["id"]] for d in test_docs if d["domain"] == "CL"}),
                        ("IT", {d["id"]: allp[d["id"]] for d in test_docs if d["domain"] == "IT"})]:
        json.dump(sel, open(f"{P3_DIR}/prediction_{name}_{suffix}.json", "w"), ensure_ascii=False)
    pickle.dump(P, open(f"{P3_DIR}/probs_test_{name}.pkl", "wb"))
    print(f"Saved prediction_{name}_{{all,CL,IT}}.json ({len(allp)} docs, {sum(map(len, allp.values()))} sentences)")

# ---------------- trainer (optional distillation from a teacher RRModel) ----------------
def train_rr(model, name, mode="document", teacher=None, alpha=0.0, T=2.0, epochs=10, patience=3,
             backbone_lr=1e-4, head_lr=1e-3, bw=None, weight_decay=0.01, warmup_frac=0.06, debug=False, seed=1):
    random.seed(seed); torch.manual_seed(seed); bw = bw or BW[mode]
    tw = all_windows(train_docs, mode); vdocs = val_docs
    if debug:
        epochs, name = 1, name + "_debug"; tw = tw[:bw * 4]
        vdocs = [next(d for d in val_docs if d["domain"] == dm) for dm in ("CL", "IT")]
    model.to(device)
    if teacher is not None:
        teacher.to(device).eval()
        for p in teacher.parameters(): p.requires_grad_(False)
    groups = {}
    for n_, p in model.named_parameters():
        if p.requires_grad: groups.setdefault((n_.startswith("head."), p.dim() >= 2), []).append(p)
    opt = torch.optim.AdamW([{"params": ps, "lr": head_lr if h else backbone_lr, "weight_decay": weight_decay if dec else 0.0}
                             for (h, dec), ps in groups.items()], betas=(0.9, 0.98))
    spe = math.ceil(len(tw) / bw); total = spe * epochs; warm = max(1, int(warmup_frac * total))
    sched = LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{name}] mode={mode} windows={len(tw)} trainable={n_train:,} {'KD alpha=%.2f T=%.1f' % (alpha, T) if teacher is not None else ''}")
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    best, best_state, bad, hist = -1.0, None, 0, []
    cw = CLASS_W if USE_CLASS_WEIGHTS else None
    for ep in range(epochs):
        model.train(); random.shuffle(tw); tl, nb = 0.0, 0
        for i in range(0, len(tw), bw):
            x, spans, y = collate(tw[i:i + bw], train_docs); x, y = x.to(device), y.to(device)
            with amp():
                lg = model(x, spans).float()
                loss = F.cross_entropy(lg, y, weight=cw)
                if teacher is not None:
                    with torch.no_grad(): tlg = teacher(x, spans).float()
                    kd = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(tlg / T, -1), reduction="batchmean") * T * T
                    loss = (1 - alpha) * loss + alpha * kd
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt.step(); sched.step(); tl += loss.item(); nb += 1
        res, _ = evaluate(model, vdocs, mode)
        hist.append(dict(epoch=ep + 1, loss=tl / nb, macro_f1=res["macro_f1"], **{f"f1_{k}": v["macro_f1"] for k, v in res["by_domain"].items()}))
        print(f"  ep{ep+1:2d} loss {tl/nb:.4f} | val macro-F1 {res['macro_f1']:.4f} "
              + " ".join(f"{k} {v['macro_f1']:.3f}" for k, v in res["by_domain"].items()) + f" | {(time.time()-t0)/60:.1f} min")
        if res["macro_f1"] > best:
            best, bad = res["macro_f1"], 0
            names = {n_ for n_, p in model.named_parameters() if p.requires_grad}
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}
        else:
            bad += 1
            if bad >= patience: print("  early stop"); break
    model.load_state_dict(best_state, strict=False)
    path = f"{P3_DIR}/rr_{name}_best.pt"; torch.save(best_state, path)
    res, P = evaluate(model, vdocs, mode)
    print(f"[{name}] BEST on dev:"); print_rr(res)
    return dict(res=res, history=hist, best_path=path, minutes=(time.time() - t0) / 60,
                peak_gb=torch.cuda.max_memory_allocated() / 1e9, trainable_params=n_train, val_probs=P)

# ---------------- sanity checks (no model needed) ----------------
for mode in CTX:
    W = all_windows(train_docs, mode)
    cover = collections.defaultdict(set)
    for di, a, b in W: cover[di].update(range(a, b))
    assert all(len(cover[di]) == len(d["sent_ids"]) for di, d in enumerate(train_docs)), f"{mode}: some sentences not covered"
    mx = max(sum(len(train_docs[di]["sent_ids"][k]) for k in range(a, b)) for di, a, b in W)
    print(f"{mode:9s} windows={len(W):5d} | every sentence covered | longest window {mx} tokens (<= {WINDOW})")
    assert mx <= WINDOW
tt = sorted(sum(map(len, d["sent_ids"])) for d in test_docs)
print("test tokens/doc median", tt[len(tt) // 2], "max", tt[-1], "| val labels:", collections.Counter(LABELS[y] for d in val_docs for y in d["y"]).most_common(4), "...")
print("✅ Phase-3 toolkit ready")

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

train docs=80 sents=16845 CL=40 IT=40
val   docs=10 sents=2142 CL=5 IT=5
test  docs=10 sents=2197 CL=5 IT=5
class weights: {'Fact': 0.56, 'Issue': 3.48, 'ArgumentPetitioner': 0.9, 'ArgumentRespondent': 0.98, 'Statute': 1.09, 'Dissent': 2.14, 'PrecedentReliedUpon': 0.82, 'PrecedentNotReliedUpon': 1.51, 'PrecedentOverruled': 10.0, 'RulingByLowerCourt': 1.22, 'RatioOfTheDecision': 0.55, 'RulingByPresentCourt': 2.05, 'None': 2.09}
sentence  windows=16845 | every sentence covered | longest window 129 tokens (<= 1024)
local8    windows= 3694 | every sentence covered | longest window 825 tokens (<= 1024)
document  windows=  892 | every sentence covered | longest window 1024 tokens (<= 1024)
test tokens/doc median 7457 max 35993 | val labels: [('Fact', 444), ('RatioOfTheDecision', 403), ('ArgumentPetitioner', 400), ('ArgumentRespondent', 272)] ...
✅ Phase-3 toolkit ready


In [ ]:
DEBUG_MODE = False    # True first, then False
MLA_CKPT = f"{DRIVE_PROJECT_DIR}/DeepSeek/deepseek_mla_85m_pretrained.pt"
assert os.path.exists(MLA_CKPT), MLA_CKPT
assert hasattr(CustomDecoderLM(32, 64, 1, 1, 128, 64).layers[0].attn, "kv_up_proj"), "this is not the MLA model class"
try: del model
except NameError: pass
MLA_TARGETS = ("q_up_proj", "kv_down_proj", "kv_up_proj", "out_proj")

def build_mla_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          max_seq_len=4096, qk_rope_head_dim=64, tie_weights=True)
    ck = torch.load(MLA_CKPT, map_location="cpu", weights_only=False)
    sd = ck.get("model", ck.get("model_state_dict", ck)) if isinstance(ck, dict) else ck
    sd = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}
    lm_.load_state_dict(sd)
    return RRModel(lm_, 768, NC, dropout).to(device)

class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x): return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMAPlusLinear(nn.Module):
    """W = ((alpha/r) B A + I) W0 on the output side; identity at init, mergeable."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x); return y + self.scale * F.linear(F.linear(y, self.A), self.B)

def inject(rr, cls, r, alpha):
    for p in rr.parameters(): p.requires_grad_(False)
    for layer in rr.lm.layers:
        for nm in MLA_TARGETS:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))
    for p in rr.head.parameters(): p.requires_grad_(True)

MLA = {}
for tag, cls, r, alpha, lr, ep, pat in [("full", None, 0, 0, 1e-4, 10, 3),
                                        ("lora", LoRALinear, 8, 16, 2e-3, 20, 5),
                                        ("lorma", LoRMAPlusLinear, 6, 12, 2e-3, 20, 5)]:
    random.seed(1); torch.manual_seed(1)
    m = build_mla_rr()
    if cls is not None:
        xb, sp, _ = collate(all_windows(val_docs, "sentence")[:4], val_docs); xb = xb.to(device)
        m.eval()
        with torch.no_grad(): before = m(xb, sp)
        inject(m, cls, r, alpha)
        with torch.no_grad(): d = (before - m(xb, sp)).abs().max().item()
        assert d < 1e-3, f"{tag}: not identity at init ({d})"; print(f"[{tag}] identity at init OK ({d:.1e})")
    MLA[tag] = train_rr(m, f"mla_{tag}_document", mode="document", epochs=ep, patience=pat,
                        backbone_lr=lr, head_lr=1e-3, debug=DEBUG_MODE)
    if not DEBUG_MODE: write_test_predictions(m, f"mla_{tag}_document", "document")
    del m; torch.cuda.empty_cache()

print(f"\n{'run':8s} {'F1':>7s} {'CL':>7s} {'IT':>7s} {'P':>7s} {'R':>7s} {'trainable':>11s} {'min':>5s} {'GB':>5s}")
for k, o in MLA.items():
    r_ = o["res"]; bd = r_["by_domain"]
    print(f"{k:8s} {r_['macro_f1']:7.4f} {bd['CL']['macro_f1']:7.4f} {bd['IT']['macro_f1']:7.4f} {r_['macro_p']:7.4f} {r_['macro_r']:7.4f} {o['trainable_params']:11,d} {o['minutes']:5.1f} {o['peak_gb']:5.1f}")

[mla_full_document] mode=document windows=892 trainable=87,038,989 
  ep 1 loss 2.8840 | val macro-F1 0.1211 CL 0.134 IT 0.117 | 0.3 min
  ep 2 loss 2.6272 | val macro-F1 0.0747 CL 0.064 IT 0.081 | 0.5 min
  ep 3 loss 1.9527 | val macro-F1 0.1791 CL 0.184 IT 0.183 | 0.8 min
  ep 4 loss 1.5164 | val macro-F1 0.2783 CL 0.275 IT 0.272 | 1.1 min
  ep 5 loss 1.1243 | val macro-F1 0.2912 CL 0.238 IT 0.291 | 1.3 min
  ep 6 loss 0.7739 | val macro-F1 0.3123 CL 0.255 IT 0.341 | 1.6 min
  ep 7 loss 0.4879 | val macro-F1 0.3176 CL 0.259 IT 0.319 | 1.9 min
  ep 8 loss 0.2919 | val macro-F1 0.3695 CL 0.305 IT 0.369 | 2.1 min
  ep 9 loss 0.1564 | val macro-F1 0.3649 CL 0.306 IT 0.358 | 2.4 min
  ep10 loss 0.0842 | val macro-F1 0.3492 CL 0.300 IT 0.343 | 2.7 min
[mla_full_document] BEST on dev:
ALL  macro-F1 0.3695 | P 0.4071 | R 0.3617 | acc 0.4225
  CL  macro-F1 0.3054 | P 0.3178 | R 0.3143 | acc 0.3968
  IT  macro-F1 0.3685 | P 0.4400 | R 0.3544 | acc 0.4705
Saved prediction_mla_full_document_{all